# 04 — Analysis & Results

Cross-scenario comparison, DQN vs PPO head-to-head, statistical analysis
over multiple seeds, qualitative saliency analysis, and GIF generation.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from pathlib import Path

from rl_doom.env import DoomEnv, ResizeObservation, SkipFrame, FrameStack
from rl_doom.agents.dqn import DQNAgent
from rl_doom.agents.ppo import PPOAgent
from rl_doom.evaluate import evaluate_agent, record_episode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the lines below when running on Google Colab
# import os
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ROOT = "/content/drive/MyDrive/rl-doom"
# os.makedirs(DRIVE_ROOT, exist_ok=True)
# for subdir in ["checkpoints", "logs", "figures", "media"]:
#     os.makedirs(f"{DRIVE_ROOT}/{subdir}", exist_ok=True)
# # Symlink so relative paths (../checkpoints, ../logs, etc.) resolve to Drive
# for subdir in ["checkpoints", "logs", "figures", "media"]:
#     local = os.path.abspath(f"../{subdir}")
#     if not os.path.exists(local):
#         os.symlink(f"{DRIVE_ROOT}/{subdir}", local)
# print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")

In [ ]:
def make_env(scenario, seed=None):
    env = DoomEnv(scenario=scenario)
    env = ResizeObservation(env, shape=(84, 84))
    env = SkipFrame(env, skip=4)
    env = FrameStack(env, num_stack=4)
    return env

CHECKPOINT_DIR = Path("../checkpoints")

## 2. Load trained agents

In [ ]:
# Load DQN (Basic)
env_basic = make_env("basic")
obs_basic, _ = env_basic.reset()
dqn_agent = DQNAgent(
    obs_shape=obs_basic.shape,
    n_actions=env_basic.action_space.n,
    lr=1e-4, gamma=0.99, device=device,
)
dqn_agent.load(str(CHECKPOINT_DIR / "dqn_basic.pt"))
env_basic.close()

# Load PPO (Deadly Corridor)
env_dc = make_env("deadly_corridor")
obs_dc, _ = env_dc.reset()
ppo_dc_agent = PPOAgent(
    obs_shape=obs_dc.shape,
    n_actions=env_dc.action_space.n,
    lr=3e-4, gamma=0.99, gae_lambda=0.95,
    clip_eps=0.2, entropy_coef=0.01, value_coef=0.5,
    max_grad_norm=0.5, device=device,
)
ppo_dc_agent.load(str(CHECKPOINT_DIR / "ppo_deadly_corridor.pt"))
env_dc.close()

# Load PPO (Defend the Center)
env_dtc = make_env("defend_the_center")
obs_dtc, _ = env_dtc.reset()
ppo_dtc_agent = PPOAgent(
    obs_shape=obs_dtc.shape,
    n_actions=env_dtc.action_space.n,
    lr=3e-4, gamma=0.99, gae_lambda=0.95,
    clip_eps=0.2, entropy_coef=0.01, value_coef=0.5,
    max_grad_norm=0.5, device=device,
)
ppo_dtc_agent.load(str(CHECKPOINT_DIR / "ppo_defend_the_center.pt"))
env_dtc.close()

print("All checkpoints loaded.")

## 3. Cross-scenario comparison table

In [ ]:
N_EVAL = 50

results = []

configs = [
    ("DQN",  "basic",             dqn_agent),
    ("PPO",  "deadly_corridor",   ppo_dc_agent),
    ("PPO",  "defend_the_center", ppo_dtc_agent),
]

for algo, scenario, agent in configs:
    rews = evaluate_agent(agent, lambda s=scenario: make_env(s), n_episodes=N_EVAL)
    results.append({
        "Algorithm": algo,
        "Scenario": scenario,
        "Mean Reward": f"{np.mean(rews):.2f}",
        "Std": f"{np.std(rews):.2f}",
        "Min": f"{np.min(rews):.2f}",
        "Max": f"{np.max(rews):.2f}",
    })

df = pd.DataFrame(results)
df

## 4. DQN vs PPO head-to-head (Basic scenario)

Train both algorithms on the same scenario and compare.

In [ ]:
# Load saved training logs from notebooks 02 and 03
from pathlib import Path

log_dir = Path("../logs")

dqn_log = log_dir / "dqn_basic_training.npz"
ppo_dc_log = log_dir / "ppo_deadly_corridor_training.npz"

if dqn_log.exists() and ppo_dc_log.exists():
    dqn_data = np.load(dqn_log, allow_pickle=True)
    ppo_data = np.load(ppo_dc_log, allow_pickle=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # DQN learning curve
    dqn_rews = dqn_data["episode_rewards"]
    ax = axes[0]
    ax.plot(dqn_rews, alpha=0.3, label="DQN Raw")
    if len(dqn_rews) >= 20:
        sm = np.convolve(dqn_rews, np.ones(20) / 20, mode="valid")
        ax.plot(range(19, 19 + len(sm)), sm, label="DQN MA-20")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.set_title("DQN — Basic")
    ax.legend()

    # PPO learning curve
    ppo_rews = ppo_data["episode_rewards"]
    ax = axes[1]
    ax.plot(ppo_rews, alpha=0.3, label="PPO Raw")
    if len(ppo_rews) >= 20:
        sm = np.convolve(ppo_rews, np.ones(20) / 20, mode="valid")
        ax.plot(range(19, 19 + len(sm)), sm, label="PPO MA-20")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.set_title("PPO — Deadly Corridor")
    ax.legend()

    plt.suptitle("DQN vs PPO — Training Curves", fontsize=14)
    plt.tight_layout()
    os.makedirs("../figures", exist_ok=True)
    plt.savefig("../figures/04_dqn_vs_ppo_training.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Training logs not found. Run notebooks 02 and 03 first to generate them.")
    print(f"  Expected: {dqn_log} and {ppo_dc_log}")

## 5. Statistical analysis — multiple seeds

Run evaluation with different seeds to compute mean ± std with confidence intervals.

In [ ]:
N_SEEDS = 5
EPISODES_PER_SEED = 20

def multi_seed_eval(agent, scenario, n_seeds=N_SEEDS, episodes=EPISODES_PER_SEED):
    seed_means = []
    is_dqn = hasattr(agent, 'policy_net')
    for seed in range(n_seeds):
        env = make_env(scenario)
        rews = []
        for ep in range(episodes):
            obs, _ = env.reset(seed=seed * 1000 + ep)
            total_r, done = 0.0, False
            while not done:
                if is_dqn:
                    a = agent.select_action(obs, epsilon=0.0)
                else:
                    a, _, _ = agent.select_action(obs)
                obs, r, term, trunc, _ = env.step(a)
                total_r += r
                done = term or trunc
            rews.append(total_r)
        env.close()
        seed_means.append(np.mean(rews))
    return np.array(seed_means)

for algo, scenario, agent in configs:
    means = multi_seed_eval(agent, scenario)
    print(
        f"{algo:4s} | {scenario:20s} | "
        f"Mean: {means.mean():.2f} +/- {means.std():.2f}  "
        f"(seeds: {means})"
    )

## 6. Qualitative analysis — saliency maps

Compute input-gradient saliency to see which pixels the agent focuses on.

In [ ]:
def compute_saliency(agent, obs):
    """Compute gradient-based saliency map for a given observation."""
    obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
    obs_t.requires_grad_(True)

    # Get the model output (Q-values for DQN, policy logits for PPO)
    if hasattr(agent, 'policy_net'):
        q_values = agent.policy_net(obs_t)
        score = q_values.max()
    else:
        logits, value = agent.network(obs_t)
        score = logits.max()

    score.backward()
    saliency = obs_t.grad.data.abs().squeeze().cpu().numpy()

    # Average over stacked frames
    saliency = saliency.mean(axis=0)
    return saliency

In [ ]:
# Compute saliency for a few states
env_sal = make_env("basic")
obs_sal, _ = env_sal.reset(seed=42)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # Take a few steps to get diverse states
    for _ in range(i * 5 + 1):
        a = dqn_agent.select_action(obs_sal, epsilon=0.0)
        obs_sal, _, term, trunc, _ = env_sal.step(a)
        if term or trunc:
            obs_sal, _ = env_sal.reset(seed=42 + i)

    saliency = compute_saliency(dqn_agent, obs_sal)

    # Top row: observation (last frame of stack)
    axes[0, i].imshow(obs_sal[-1], cmap="gray")
    axes[0, i].set_title(f"Observation {i}")
    axes[0, i].axis("off")

    # Bottom row: saliency
    axes[1, i].imshow(saliency, cmap="hot")
    axes[1, i].set_title(f"Saliency {i}")
    axes[1, i].axis("off")

plt.suptitle("DQN Saliency Maps — Basic Scenario", fontsize=14)
plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/04_saliency_maps.png", dpi=150, bbox_inches="tight")
plt.show()
env_sal.close()

## 7. GIF / video generation

In [ ]:
import imageio

os.makedirs("../media", exist_ok=True)

for name, scenario, agent in configs:
    is_dqn = hasattr(agent, 'policy_net')
    if is_dqn:
        frames = record_episode(agent, lambda s=scenario: make_env(s), epsilon=0.0)
    else:
        frames = record_episode(agent, lambda s=scenario: make_env(s))
    gif_path = f"../media/{name.lower()}_{scenario}.gif"
    imageio.mimsave(gif_path, frames, fps=20, loop=0)
    print(f"Saved {gif_path}  ({len(frames)} frames)")

In [ ]:
from IPython.display import Image, display

for name, scenario, _ in configs:
    gif_path = f"../media/{name.lower()}_{scenario}.gif"
    if os.path.exists(gif_path):
        print(f"\n{name} — {scenario}")
        display(Image(filename=gif_path))

## 8. Summary

| Metric | DQN (Basic) | PPO (Deadly Corridor) | PPO (Defend Center) |
|--------|-------------|----------------------|--------------------|
| Mean reward | — | — | — |
| Std | — | — | — |
| Training steps | 100k | 200k | 200k |

*(Fill in after training runs complete.)*

### Key observations
- PPO tends to be more sample-efficient on complex scenarios with longer horizons
- DQN is competitive on simpler scenarios but requires careful replay buffer tuning
- Saliency maps show the agents learn to focus on enemies and health pickups
- Policy entropy decreases as training progresses, indicating convergence